In [1]:
# ==========================================
# BERT Sentiment Analysis using Hugging Face
# ==========================================

import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ==========================================
# Check GPU
# ==========================================

print("=" * 50)
print("PyTorch Version:", torch.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("=" * 50)

# ==========================================
# Load Dataset
# ==========================================

print("\nLoading IMDb Dataset...")

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

# ==========================================
# Reduce Dataset (for quick training)
# ==========================================

train_dataset = dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

# ==========================================
# Load Tokenizer
# ==========================================

print("\nLoading BERT Tokenizer...")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# ==========================================
# Tokenization Function
# ==========================================

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("\nTokenizing Dataset...")

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# ==========================================
# Format Dataset
# ==========================================

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

# ==========================================
# Load BERT Model
# ==========================================

print("\nLoading Pretrained BERT Model...")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# ==========================================
# Evaluation Metrics
# ==========================================

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# ==========================================
# Training Arguments
# ==========================================

training_args = TrainingArguments(

    output_dir="./bert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    report_to="none"
)

# ==========================================
# Trainer
# ==========================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

# ==========================================
# Train Model
# ==========================================

print("\nStarting Training...\n")

trainer.train()

# ==========================================
# Evaluate
# ==========================================

print("\nEvaluating Model...\n")

results = trainer.evaluate()

print(results)

# ==========================================
# Save Model
# ==========================================

print("\nSaving Model...\n")

trainer.save_model("bert_sentiment_model")

tokenizer.save_pretrained("bert_sentiment_model")

print("Model Saved Successfully")

# ==========================================
# Load Saved Model
# ==========================================

classifier = pipeline(

    "sentiment-analysis",

    model="bert_sentiment_model",

    tokenizer="bert_sentiment_model",

    device=0 if torch.cuda.is_available() else -1
)

# ==========================================
# Test Custom Sentences
# ==========================================

texts = [

    "This movie is fantastic. I loved it.",

    "Worst movie ever made.",

    "The acting was average.",

    "Excellent direction and amazing screenplay.",

    "I will never watch this again."
]

print("\nPredictions\n")

for sentence in texts:

    prediction = classifier(sentence)[0]

    print("-" * 60)

    print("Sentence :", sentence)

    print("Prediction :", prediction["label"])

    print("Confidence :", round(prediction["score"] * 100, 2), "%")

print("-" * 60)

print("\nCompleted Successfully")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch Version: 2.5.0a0+e000cf0ad9.nv24.10
Using Device: cuda
GPU: NVIDIA H200 MIG 1g.18gb

Loading IMDb Dataset...


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Loading BERT Tokenizer...

Tokenizing Dataset...

Loading Pretrained BERT Model...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 11303.72it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were


Starting Training...



...

{'eval_loss': 0.3948596119880676, 'eval_accuracy': 0.859, 'eval_precision': 0.8577319587628865, 'eval_recall': 0.8524590163934426, 'eval_f1': 0.8550873586844809}

Saving Model...



Writing model shards: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.13it/s]


Model Saved Successfully


Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 12224.21it/s]


Predictions

------------------------------------------------------------
Sentence : This movie is fantastic. I loved it.
Prediction : LABEL_1
Confidence : 97.65 %
------------------------------------------------------------
Sentence : Worst movie ever made.
Prediction : LABEL_0
Confidence : 97.83 %
------------------------------------------------------------
Sentence : The acting was average.
Prediction : LABEL_0
Confidence : 92.72 %
------------------------------------------------------------
Sentence : Excellent direction and amazing screenplay.
Prediction : LABEL_1
Confidence : 97.38 %
------------------------------------------------------------
Sentence : I will never watch this again.
Prediction : LABEL_0
Confidence : 92.32 %
------------------------------------------------------------

Completed Successfully
